# Sequence-to-Sequence Learning with an LSTM Encoder–Decoder

A **sequence-to-sequence (seq2seq)** model maps one variable-length sequence to another. Examples include machine translation, summarization, dialogue, and converting speech into text.

In this notebook we build a small German-to-English translator from first principles with PyTorch. The dataset is deliberately small and self-contained: the objective is to understand the architecture and tensor flow, not to produce a production-quality translator.

### Learning objectives

By the end, you should be able to explain and implement:

- source and target sequences, special tokens, vocabularies, and padding;
- an LSTM encoder that compresses a source sequence into context states;
- an LSTM decoder that generates one target token at a time;
- teacher forcing during training and autoregressive decoding during inference;
- masked cross-entropy loss, gradient clipping, validation, and greedy translation; and
- the limitations of a fixed-context encoder–decoder and why attention is useful.

> Recommended prerequisites: `RNNs.md`, `GRUs.ipynb`, and `LSTMs.ipynb`.

## 1. The sequence-to-sequence task

Let the source sequence be

$$X=(x_1,x_2,\ldots,x_{T_x})$$

and the target sequence be

$$Y=(y_1,y_2,\ldots,y_{T_y}).$$

The lengths $T_x$ and $T_y$ do not need to be equal. For example:

```text
source (German):  heute sehe ich das haus
target (English): today i see the house
```

We add boundary tokens:

```text
<sos> heute sehe ich das haus <eos>
<sos> today i see the house <eos>
```

`<sos>` tells the decoder to begin. `<eos>` teaches it when to stop. `<pad>` makes sequences in a batch equally long, and `<unk>` represents tokens absent from the training vocabulary.

## 2. Encoder–decoder architecture

The model contains two recurrent networks:

1. The **encoder** reads the complete source sequence.
2. The **decoder** uses the encoder's final state to generate the target sequence one token at a time.

```text
source token IDs
      │
      ▼
source embeddings → encoder LSTM → (h_T, c_T)
                                      │
                              context │ states
                                      ▼
<sos> → decoder LSTM → token logits → predicted token
              ▲                              │
              └──────────────────────────────┘
                    next decoding step
```

For an LSTM, the context is a pair:

$$z=(h_{T_x},c_{T_x}).$$

$h_{T_x}$ is the final hidden state and $c_{T_x}$ is the final cell state. They initialize the decoder. This first architecture passes only the final states to the decoder; later we discuss why compressing the whole source into fixed-size states becomes a bottleneck.

## 3. Setup and reproducibility

Fixing random seeds makes this small demonstration easier to reproduce. Exact results can still vary across PyTorch versions and hardware.

In [ ]:
import copy
import math
import random
from collections import Counter

import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_sequence
from torch.utils.data import DataLoader, Dataset

SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

## 4. Parallel text data

Translation training requires **parallel data**: each source sentence is paired with its target translation. Here we generate a tiny, regular corpus so the entire notebook works offline. Real translation systems use much larger and more diverse corpora.

In [ ]:
nouns = {
    "auto": "car",
    "buch": "book",
    "haus": "house",
    "boot": "boat",
}
colors = {
    "rot": "red",
    "blau": "blue",
    "grün": "green",
    "gelb": "yellow",
}

pairs = []
for de_noun, en_noun in nouns.items():
    for de_color, en_color in colors.items():
        pairs.append((f"das {de_noun} ist {de_color}", f"the {en_noun} is {en_color}"))

    pairs.extend(
        [
            (f"ich sehe das {de_noun}", f"i see the {en_noun}"),
            (f"heute sehe ich das {de_noun}", f"today i see the {en_noun}"),
            (f"morgen sehe ich das {de_noun}", f"tomorrow i see the {en_noun}"),
            (f"wir haben das {de_noun}", f"we have the {en_noun}"),
        ]
    )

random.shuffle(pairs)
train_pairs = pairs[:24]
valid_pairs = pairs[24:28]
test_pairs = pairs[28:]

print(f"train={len(train_pairs)}, valid={len(valid_pairs)}, test={len(test_pairs)}")
for source, target in train_pairs[:3]:
    print(f"{source:28s} -> {target}")

## 5. Tokenization and vocabularies

A tokenizer converts text into tokens. A vocabulary then maps tokens to integer IDs because neural networks operate on numbers, not strings. Source and target languages need separate vocabularies and embedding layers.

Vocabularies should be built from the **training split only**. Including validation or test text would leak information about evaluation data. Our whitespace tokenizer is sufficient for this controlled corpus; real data benefits from a proper tokenizer or a subword method such as BPE/SentencePiece.

In [ ]:
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"
SPECIAL_TOKENS = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]

def tokenize(text):
    return text.lower().split()

def build_vocab(sentences, min_frequency=1):
    counts = Counter(token for sentence in sentences for token in tokenize(sentence))
    tokens = SPECIAL_TOKENS + sorted(
        token for token, count in counts.items() if count >= min_frequency
    )
    stoi = {token: index for index, token in enumerate(tokens)}
    return stoi, tokens

source_stoi, source_itos = build_vocab(source for source, _ in train_pairs)
target_stoi, target_itos = build_vocab(target for _, target in train_pairs)

PAD_IDX = target_stoi[PAD_TOKEN]  # special-token indices match in both vocabularies
SOS_IDX = target_stoi[SOS_TOKEN]
EOS_IDX = target_stoi[EOS_TOKEN]

print("source vocabulary size:", len(source_itos))
print("target vocabulary size:", len(target_itos))
print("special-token IDs:", {token: target_stoi[token] for token in SPECIAL_TOKENS})

## 6. Numericalization, padding, and batching

**Numericalization** converts tokens to vocabulary IDs. A batch requires a rectangular tensor, so shorter sequences are padded to the longest sequence in that batch.

With `batch_first=True`, our tensors have shapes:

| Tensor | Shape |
|---|---|
| Source IDs | $(B,T_x)$ |
| Target IDs | $(B,T_y)$ |
| Source lengths | $(B,)$ |

We retain the true source lengths so the encoder can use a packed sequence and avoid treating padding as real input.

In [ ]:
def numericalize(text, stoi):
    unk_index = stoi[UNK_TOKEN]
    token_ids = [stoi[SOS_TOKEN]]
    token_ids += [stoi.get(token, unk_index) for token in tokenize(text)]
    token_ids.append(stoi[EOS_TOKEN])
    return torch.tensor(token_ids, dtype=torch.long)

class TranslationDataset(Dataset):
    def __init__(self, sentence_pairs):
        self.sentence_pairs = sentence_pairs

    def __len__(self):
        return len(self.sentence_pairs)

    def __getitem__(self, index):
        source, target = self.sentence_pairs[index]
        return numericalize(source, source_stoi), numericalize(target, target_stoi)

def collate_batch(examples):
    source_sequences, target_sequences = zip(*examples)
    source_lengths = torch.tensor([len(sequence) for sequence in source_sequences])
    source_batch = pad_sequence(source_sequences, batch_first=True, padding_value=PAD_IDX)
    target_batch = pad_sequence(target_sequences, batch_first=True, padding_value=PAD_IDX)
    return source_batch, source_lengths, target_batch

def make_loader(sentence_pairs, shuffle=False, batch_size=8):
    return DataLoader(
        TranslationDataset(sentence_pairs),
        batch_size=batch_size,
        shuffle=shuffle,
        collate_fn=collate_batch,
    )

train_loader = make_loader(train_pairs, shuffle=True)
valid_loader = make_loader(valid_pairs)
test_loader = make_loader(test_pairs)

In [ ]:
source_batch, source_lengths, target_batch = next(iter(train_loader))
print("source batch:  ", source_batch.shape)
print("source lengths:", source_lengths.shape, source_lengths.tolist())
print("target batch:  ", target_batch.shape)
print("first source IDs:", source_batch[0].tolist())
print("first source tokens:", [source_itos[i] for i in source_batch[0].tolist()])

## 7. The LSTM encoder

The encoder performs two operations:

1. An embedding layer maps each source token ID to a dense vector.
2. An LSTM reads the embedding sequence and returns final states from every layer.

For $L$ layers and hidden size $n_h$:

| Value | Shape |
|---|---|
| Embedded source | $(B,T_x,n_e)$ |
| Final hidden state $h_T$ | $(L,B,n_h)$ |
| Final cell state $c_T$ | $(L,B,n_h)$ |

We do not need all encoder outputs in this fixed-context model, so the encoder returns only `(hidden, cell)`.

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, embedding_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(input_dim, embedding_dim, padding_idx=PAD_IDX)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )

    def forward(self, source, source_lengths):
        # source: (B, T_x)
        embedded = self.dropout(self.embedding(source))  # (B, T_x, n_e)
        packed = pack_padded_sequence(
            embedded, source_lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, cell) = self.lstm(packed)
        # hidden and cell: (L, B, n_h)
        return hidden, cell

## 8. The LSTM decoder

The decoder handles one token per call. Given the current input token and previous LSTM states, it produces:

- logits over the target vocabulary;
- the next hidden state; and
- the next cell state.

$$
(y_t,h_{t-1},c_{t-1})\longrightarrow(\text{logits}_{t+1},h_t,c_t).
$$

The linear layer converts the top decoder layer's output from hidden size $n_h$ to target-vocabulary size $|V_y|$. These are raw logits; `CrossEntropyLoss` applies the required log-softmax internally.

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, embedding_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.output_dim = output_dim
        self.embedding = nn.Embedding(output_dim, embedding_dim, padding_idx=PAD_IDX)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=num_layers,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=True,
        )
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, input_token, hidden, cell):
        # input_token: (B,) -> add a one-step time dimension
        embedded = self.dropout(self.embedding(input_token)).unsqueeze(1)  # (B, 1, n_e)
        output, (hidden, cell) = self.lstm(embedded, (hidden, cell))
        logits = self.output_layer(output.squeeze(1))  # (B, |V_y|)
        return logits, hidden, cell

## 9. Joining the models and teacher forcing

The complete model first encodes the source and then loops over target positions. After predicting a token, it must choose the next decoder input:

- **Teacher forcing:** use the correct target token. This provides a clean learning signal.
- **Autoregressive feedback:** use the model's own predicted token. This matches inference but can compound early errors.

A teacher-forcing ratio of 0.5 chooses the ground-truth token with probability 0.5 for each example and step. Evaluation uses a ratio of 0.0. The difference between training on correct histories and predicting from imperfect histories is often called **exposure bias**.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, source, source_lengths, target, teacher_forcing_ratio=0.5):
        batch_size, target_length = target.shape
        outputs = torch.zeros(
            batch_size, target_length, self.decoder.output_dim, device=source.device
        )

        hidden, cell = self.encoder(source, source_lengths)
        input_token = target[:, 0]  # every target begins with <sos>

        for t in range(1, target_length):
            logits, hidden, cell = self.decoder(input_token, hidden, cell)
            outputs[:, t] = logits
            predicted_token = logits.argmax(dim=1)
            use_teacher = torch.rand(batch_size, device=source.device) < teacher_forcing_ratio
            input_token = torch.where(use_teacher, target[:, t], predicted_token)

        return outputs

## 10. Initialize the model and inspect tensor flow

The encoder and decoder must use compatible hidden sizes and layer counts because the encoder states directly initialize the decoder. Their embedding sizes may differ.

In [ ]:
EMBEDDING_DIM = 32
HIDDEN_DIM = 64
NUM_LAYERS = 2
DROPOUT = 0.2

encoder = Encoder(len(source_itos), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
decoder = Decoder(len(target_itos), EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT)
model = Seq2Seq(encoder, decoder).to(device)

def initialize_weights(module):
    if isinstance(module, (nn.Linear, nn.Embedding)):
        nn.init.uniform_(module.weight, -0.08, 0.08)
    if isinstance(module, nn.Linear) and module.bias is not None:
        nn.init.zeros_(module.bias)

model.apply(initialize_weights)

parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print(f"trainable parameters: {parameter_count:,}")

source_batch, source_lengths, target_batch = next(iter(train_loader))
source_batch = source_batch.to(device)
target_batch = target_batch.to(device)
with torch.no_grad():
    output = model(source_batch, source_lengths, target_batch, teacher_forcing_ratio=0.0)
print("model output shape:", output.shape, "= (batch, target length, target vocabulary)")

## 11. Loss, optimization, and training

At each non-initial target position, the model predicts a categorical distribution over the target vocabulary. We flatten the time and batch dimensions before applying cross-entropy.

The `<sos>` position is excluded because the decoder receives it rather than predicts it. Padding is ignored with `ignore_index=PAD_IDX`. Gradient clipping limits exploding gradients, which can still affect gated RNNs.

In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

def run_epoch(model, loader, optimizer=None, teacher_forcing_ratio=0.0, clip=1.0):
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0

    for source, source_lengths, target in loader:
        source, target = source.to(device), target.to(device)

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            output = model(source, source_lengths, target, teacher_forcing_ratio)
            logits = output[:, 1:].reshape(-1, output.size(-1))
            expected = target[:, 1:].reshape(-1)
            loss = criterion(logits, expected)

            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), clip)
                optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
EPOCHS = 60
best_valid_loss = float("inf")
best_state = None

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(
        model, train_loader, optimizer, teacher_forcing_ratio=0.5, clip=1.0
    )
    valid_loss = run_epoch(model, valid_loader)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        best_state = copy.deepcopy(model.state_dict())

    if epoch == 1 or epoch % 10 == 0:
        perplexity = math.exp(min(valid_loss, 20))
        print(
            f"epoch {epoch:02d} | train loss {train_loss:.3f} | "
            f"valid loss {valid_loss:.3f} | valid perplexity {perplexity:.2f}"
        )

model.load_state_dict(best_state)

## 12. Autoregressive inference

At inference time there is no target sentence and no teacher forcing. The procedure is:

1. Encode the source once.
2. Give `<sos>` to the decoder.
3. Select the highest-logit token (greedy decoding).
4. Feed that prediction back as the next input.
5. Stop at `<eos>` or a maximum length.

Greedy decoding is simple but not guaranteed to find the highest-probability complete sequence. Beam search keeps several partial candidates and is a common extension.

In [ ]:
@torch.no_grad()
def translate(model, sentence, max_output_length=20):
    model.eval()
    source = numericalize(sentence, source_stoi).unsqueeze(0).to(device)
    source_lengths = torch.tensor([source.size(1)])
    hidden, cell = model.encoder(source, source_lengths)

    input_token = torch.tensor([SOS_IDX], device=device)
    generated_tokens = []

    for _ in range(max_output_length):
        logits, hidden, cell = model.decoder(input_token, hidden, cell)
        predicted_index = logits.argmax(dim=1).item()
        if predicted_index == EOS_IDX:
            break
        generated_tokens.append(target_itos[predicted_index])
        input_token = torch.tensor([predicted_index], device=device)

    return generated_tokens

for source, expected in test_pairs:
    prediction = " ".join(translate(model, source))
    print(f"source:   {source}")
    print(f"expected: {expected}")
    print(f"predicted:{' ' if prediction else ''}{prediction}\n")

## 13. Evaluation

Loss measures token prediction under the model, while translation quality is a sequence-level question. Useful metrics include:

- **Perplexity:** exponentiated average token loss; lower is better.
- **Token accuracy:** easy to interpret but does not reward valid alternative translations.
- **Exact match:** strict; mainly useful for controlled tasks.
- **BLEU:** compares matching n-grams with one or more reference translations.

Automatic metrics are imperfect and should be accompanied by example inspection. Our tiny synthetic split is suitable for checking the implementation, not for making claims about translation quality.

In [ ]:
test_loss = run_epoch(model, test_loader)
exact_matches = sum(
    translate(model, source) == tokenize(expected) for source, expected in test_pairs
)
print(f"test loss:       {test_loss:.3f}")
print(f"test perplexity: {math.exp(min(test_loss, 20)):.2f}")
print(f"exact match:     {exact_matches}/{len(test_pairs)}")

## 14. Limitations and the path to attention

This model forces the encoder to compress the complete source into the fixed-size pair $(h_T,c_T)$. As the source grows, details from early positions can be difficult to preserve. This is the **fixed-context bottleneck**.

An attention-based decoder receives all encoder outputs $(h_1,\ldots,h_{T_x})$ and learns which source positions to consult at each decoding step. Other limitations of this first model include:

- greedy rather than beam-search decoding;
- word-level rather than subword tokenization;
- no bidirectional encoder;
- exposure bias from teacher forcing; and
- a tiny synthetic corpus with little linguistic variety.

## 15. Scaling the same pipeline to Multi30k

To turn this lesson into a real German-to-English experiment, keep the model and training structure but replace the data layer:

1. Load train, validation, and test splits from Multi30k.
2. Use German and English tokenizers or train a shared subword tokenizer.
3. Build vocabularies from the training split only.
4. Numericalize every example and retain `<sos>`, `<eos>`, `<unk>`, and `<pad>`.
5. Batch with padding and true source lengths.
6. Increase embedding/hidden dimensions and train for more epochs.
7. Save the state with the best validation loss.
8. Report BLEU on the test split only after model choices are finalized.

The architectural lesson does not change: the encoder produces context states, the decoder generates conditionally on those states, and all parameters are trained jointly from target-token loss.

## 16. Summary and exercises

### Summary

- The encoder converts a variable-length source into final LSTM states $(h_T,c_T)$.
- Those states initialize a decoder that generates target tokens sequentially.
- Teacher forcing helps training, while inference feeds predictions back into the decoder.
- Padding must be excluded from the loss, and source lengths let the encoder ignore padded steps.
- Gradient clipping improves recurrent-training stability.
- Fixed context is a bottleneck; attention is the natural next step.

### Suggested exercises

1. Set the teacher-forcing ratio to 0 and 1. Compare convergence and inference.
2. Print `hidden.shape` and `cell.shape` inside both modules and explain every dimension.
3. Remove packed sequences and observe why padding can affect the encoder context.
4. Add token accuracy that ignores `<pad>`.
5. Replace greedy decoding with a small beam search.
6. Modify the encoder to return all outputs in preparation for attention.